In [ ]:
import uproot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Apri il file
file = uproot.open("CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_CONCAT.root")
#file = uproot.open("CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_40_PRO_KE_40_CONCAT.root")
#file = uproot.open("CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_40_PRO_KE_40_CHI2_VAR_CONCAT.root")
#file = uproot.open("CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_40_PRO_KE_40_CHI2_VAR_TRACKSCORE_VAR_CONCAT.root")
#file = uproot.open("CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_40_PRO_KE_40_CHI2_VAR_TRACKSCORE_VAR_VISIBLE_VAR_CONCAT.root")
#file = uproot.open("CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_50_PRO_KE_50_CHI2_VAR_TRACKSCORE_VAR_VISIBLE_VAR_CONCAT.root")

# Vedi cosa contiene (tree, histogram, ecc.)
print(file.keys())          # lista di oggetti nel file
print(file.classnames())    # tipo di ogni oggetto (TTree, TH1F, ecc.)

tree_nu = file["events/selectedNu"]   # sostituisci con il nome reale
print(tree_nu.keys())             # nomi delle colonne (branch)

tree_reco = file["events/selectedReco"]
print(tree_reco.keys())

df_nu = tree_nu.arrays(library="pd") # converte tutto in un pandas DataFrame
df_reco = tree_reco.arrays(library="pd")

In [ ]:
print(df_nu.columns)

In [ ]:
print(df_reco.columns)

In [ ]:

chiavi = ['Run', 'Subrun', 'Evt']

merged = df_reco.merge(df_nu[chiavi], on=chiavi, how='inner')

correctly_classified = len(merged)

efficiency = correctly_classified / df_nu.shape[0]
purity = correctly_classified / df_reco.shape[0]

print('tot selected: ', df_reco.shape[0])
print('sel. corr: ', correctly_classified)
print('tot true: ', df_nu.shape[0])
print('efficiency: ', efficiency)
print('purity: ', purity)

In [ ]:
mask = (df_reco['Reco_class'] == 1) | (df_reco['Reco_class'] == 2)
correctly_classified_slice = df_reco[mask]
efficiency_slice = correctly_classified_slice.shape[0] / df_nu.shape[0]
purity_slice = correctly_classified_slice.shape[0] / df_reco.shape[0]

print('tot selected: ', df_reco.shape[0])
print('sel. corr: ', correctly_classified_slice.shape[0])
print('tot true: ', df_nu.shape[0])
print('efficiency: ', efficiency_slice)
print('purity: ', purity_slice)
#f'tot sel. {8179}\nsel. corr. {3521}\ntot true {8130}\neff: {0.43308733087330875*100:.2f} pur: {0.43049272527203813*100:.2f}'

In [ ]:
mask = (df_reco['Number_protons'] > 1) & (df_reco['Reco_class'] == 2)
correctly_classified_slice = df_reco[mask]
efficiency_slice = correctly_classified_slice.shape[0] / df_nu[df_nu['True_protons']==1].shape[0]
purity_slice = correctly_classified_slice.shape[0] / df_reco[df_reco['Number_protons'] > 1].shape[0]

print('tot selected: ', df_reco[df_reco['Number_protons'] > 1].shape[0])
print('sel. corr: ', correctly_classified_slice.shape[0])
print('tot true: ', df_nu[df_nu['True_protons']==1].shape[0])
print('efficiency: ', efficiency_slice)
print('purity: ', purity_slice)
#f'tot sel. {1615}\nsel. corr. {924}\ntot true {3408}\neff: {0.2711267605633803*100:.2f} pur: {0.5721362229102167*100:.2f}'

In [ ]:
bin_edges = np.array([0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.25, 1.5, 2.0, 2.5])
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

def get_df(filename) : 
    file = uproot.open(filename)

    tree_nu = file["events/selectedNu"]  

    tree_reco = file["events/selectedReco"]

    df_nu = tree_nu.arrays(library="pd") 
    df_reco = tree_reco.arrays(library="pd")

    return df_nu[(df_nu['True_protons']==1)],df_reco[df_reco['Number_protons'] > 1]
    #return df_nu,df_reco

def get_eff_pur(df_reco, df_nu, color, label) :
    #mask = (df_reco['Reco_class'] == 1) | (df_reco['Reco_class'] == 2)
    #correctly_classified_slice = df_reco[mask]
    #efficiency_slice = correctly_classified_slice.shape[0] / df_nu.shape[0]
    #purity_slice = correctly_classified_slice.shape[0] / df_reco.shape[0]

    chiavi = ['Run', 'Subrun', 'Evt']

    merged = df_nu.merge(df_reco[chiavi], on=chiavi, how='inner')

    correctly_classified = len(merged)

    efficiency = correctly_classified / df_nu.shape[0]
    purity = correctly_classified / df_reco.shape[0]
    
    counts_all, _ = np.histogram(df_nu['true_Enu'], bins=bin_edges)

    counts_sel, _ = np.histogram(merged['true_Enu'], bins=bin_edges)

    ratio = np.zeros_like(counts_all, dtype=float)
    mask = counts_all > 0
    ratio[mask] = counts_sel[mask] / counts_all[mask]

    errors = np.zeros_like(ratio)
    errors[mask] = np.sqrt(ratio[mask] * (1 - ratio[mask]) / counts_all[mask])

    plt.stairs(ratio, bin_edges, color=color, linewidth=2, alpha=1, label=f'{label} \neff. : {efficiency*100:.1f}%, pur. w/out off. : {purity*100:.1f}%')

    # Plot error bars
    plt.errorbar(bin_centers, ratio, yerr=errors, fmt='o', markersize=3, color=color, alpha=1)


    return efficiency, purity

In [ ]:
df_nu_standard, df_reco_standard = get_df('CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_CONCAT.root')
df_nu_muL_pKE, df_reco_muL_pKE = get_df('CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_40_PRO_KE_40_CONCAT.root')
df_nu_muL_pKE_pid, df_reco_muL_pKE_pid = get_df('CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_40_PRO_KE_40_CHI2_VAR_CONCAT.root')
df_nu_muL_pKE_pid_trackscore, df_reco_muL_pKE_pid_trackscore = get_df('CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_40_PRO_KE_40_CHI2_VAR_TRACKSCORE_VAR_CONCAT.root')
df_nu_muL_pKE_pid_trackscore_vis, df_reco_muL_pKE_pid_trackscore_vis = get_df('CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_40_PRO_KE_40_CHI2_VAR_TRACKSCORE_VAR_VISIBLE_VAR_CONCAT.root')
#df_nu_muL50_pKE50_pid_trackscore_vis, df_reco_muL50_pKE50_pid_trackscore_vis = get_df('CONCAT/SBRUCE_TREE_MAPLE_NICOLA_RUN4_MU_L_40_PRO_KE_40_CHI2_VAR_TRACKSCORE_VAR_VISIBLE_VAR_CONCAT.root')

eff_standard, pur_standard = get_eff_pur(df_reco_standard,df_nu_standard,'black','orginal')
eff_muL_pKE, pur_muL_pKE = get_eff_pur(df_reco_muL_pKE, df_nu_muL_pKE, 'cornflowerblue', r'$L_{\mu}$>40 cm, $KE_p$>40 MeV')
eff_muL_pKE_pid, pur_muL_pKE_pid = get_eff_pur(df_reco_muL_pKE_pid, df_nu_muL_pKE_pid, 'red', r'$L_{\mu}$>40 cm, $KE_p$>40 MeV, GUMP $\chi^2$')
eff_muL_pKE_pid_trackscore, pur_muL_pKE_pid_trackscore = get_eff_pur(df_reco_muL_pKE_pid_trackscore, df_nu_muL_pKE_pid_trackscore, 'green', r'$L_{\mu}$>40 cm, $KE_p$>40 MeV, GUMP $\chi^2$, no $p$ trkScore')
eff_muL_pKE_pid_trackscore_vis, pur_muL_pKE_pid_trackscore_vis = get_eff_pur(df_reco_muL_pKE_pid_trackscore_vis, df_nu_muL_pKE_pid_trackscore_vis, 'orange', r'$L_{\mu}$>40 cm, $KE_p$>40 MeV, GUMP $\chi^2$, no $p$ trkScore, $\mu$/$p$ only')
#eff_muL50_pKE50_pid_trackscore_vis, pur_muL50_pKE50_pid_trackscore_vis = get_eff_pur(df_reco_muL50_pKE50_pid_trackscore_vis, df_nu_muL50_pKE50_pid_trackscore_vis, 'violet', r'$L_{\mu}$>50 cm, $KE_p$>50 MeV, GUMP $\chi^2$, no $p$ trkScore, $\mu$/$p$ only')

plt.legend(loc='lower center', fontsize=7)
plt.xlabel(r'$E_{\nu}$ true [GeV]', fontsize = 18)
plt.ylabel('efficiency', fontsize = 18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.title(r'$1\mu Np$ $N>1$ slices - ICARUS RUN 4 - $1.20\times10^{20}$POT', fontsize = 18)
plt.ylim(0,0.4)



In [ ]:
def plot_E_reco(df_reco) :
    E_1mu1p = df_reco[df_reco['Reco_class']==1]['recoE']
    E_1muNp = df_reco[df_reco['Reco_class']==2]['recoE']
    E_cosmic = df_reco[df_reco['Reco_class']==3]['recoE']
    E_oofv = df_reco[df_reco['Reco_class']==4]['recoE']
    E_nue = df_reco[df_reco['Reco_class']==5]['recoE']
    E_nc = df_reco[df_reco['Reco_class']==6]['recoE']
    E_qe = df_reco[df_reco['Reco_class']==7]['recoE']
    E_res = df_reco[df_reco['Reco_class']==8]['recoE']
    E_dis = df_reco[df_reco['Reco_class']==9]['recoE']
    E_coh = df_reco[df_reco['Reco_class']==10]['recoE']
    E_mec = df_reco[df_reco['Reco_class']==11]['recoE']
    E_other = df_reco[df_reco['Reco_class']==12]['recoE']

    frac_1mu1p = len(E_1mu1p)/df_reco.shape[0] * 100
    frac_1muNp = len(E_1muNp)/df_reco.shape[0] * 100
    frac_cosmic = len(E_cosmic)/df_reco.shape[0] * 100
    frac_oofv = len(E_oofv)/df_reco.shape[0] * 100
    frac_nue = len(E_nue)/df_reco.shape[0] * 100
    frac_nc = len(E_nc)/df_reco.shape[0] * 100
    frac_qe = len(E_qe)/df_reco.shape[0] * 100
    frac_res = len(E_res)/df_reco.shape[0] * 100
    frac_dis = len(E_dis)/df_reco.shape[0] * 100
    frac_coh = len(E_coh)/df_reco.shape[0] * 100
    frac_mec = len(E_mec)/df_reco.shape[0] * 100
    frac_other = len(E_other)/df_reco.shape[0] * 100

    plt.hist(
        [E_1mu1p,
         E_1muNp,
         E_cosmic,
         E_oofv,
         E_nue,
         E_nc,
         E_qe,
         E_res,
         E_dis,
         E_coh,
         E_mec,
         E_other
        ],
        bins= np.array([0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.25, 1.5, 2.0, 2.5]),
        histtype='bar',
        stacked=True,
        label=
        [rf'$1\mu 1p$ ({frac_1mu1p:.2f}%)',
         rf'$1\mu Np$ ({frac_1muNp:.2f}%)',
         f'cosmics ({frac_cosmic:.2f}%)',
         f'OOFV ({frac_oofv:.2f}%)',
         rf'$\nu_e$ ({frac_nue:.2f}%)',
         rf'NC ({frac_nc:.2f}%)',
         rf'QE ({frac_qe:.2f}%)',
         rf'RES ({frac_res:.2f}%)',
         rf'DIS ({frac_dis:.2f}%)',
         rf'COH ({frac_coh:.2f}%)',
         rf'MEC ({frac_mec:.2f}%)',
         rf'other ({frac_other:.2f}%)'
        ]
    )
    plt.xlabel(r'E true [GeV]', fontsize = 18)
    plt.ylabel(r'counts', fontsize = 18)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.legend(loc='upper right', fontsize=8, ncols=2)
    plt.show()

In [ ]:
#plt.title(r'Standard MAPLE selection',fontsize=18)
#plt.title(r'$L_{\mu}$>40 cm, $KE_p$>40 MeV',fontsize=18)
#plt.title(r'$L_{\mu}$>40 cm, $KE_p$>40 MeV, GUMP $\chi^2$',fontsize=18)
#plt.title(r'$L_{\mu}$>40 cm, $KE_p$>40 MeV, GUMP $\chi^2$, no $p$ trkScore',fontsize=18)
plt.title(r'$L_{\mu}$>40 cm, $KE_p$>40 MeV, GUMP $\chi^2$, no $p$ trkScore, $\mu$/$p$ only',fontsize=18)
#plot_E_reco(df_reco_standard)
#plot_E_reco(df_reco_muL_pKE)
#plot_E_reco(df_reco_muL_pKE_pid)
#plot_E_reco(df_reco_muL_pKE_pid_trackscore)
plot_E_reco(df_reco_muL_pKE_pid_trackscore_vis)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.axis("off")

rows = [
    ["standard",
     f'tot sel. {10012}\nsel. corr. {7474}\ntot true {14835}\neff: {0.5038085608358611*100:.2f} pur: {0.7465041949660407*100:.2f}',
     'tot sel. 42',
     f'tot sel. {2081}\nsel. corr. {1318}\ntot true {4257}\neff: {0.30960770495654216*100:.2f} pur: {0.6333493512734263*100:.2f}',
     'tot sel. 2',],

    ["$L_{\\mu}>40$ cm and $KE_p>40$ MeV",
     f'tot sel. {10813}\nsel. corr. {8059}\ntot true {16543}\neff: {0.48715468778335247*100:.2f} pur: {0.7453065754184778*100:.2f}',
     'tot sel. 55',
     f'tot sel. {2349}\nsel. corr. {1502}\ntot true {5342}\neff: {0.2811681018345189*100:.2f} pur: {0.6394210302256279*100:.2f}',
     'tot sel. 2'],

    ["$L_{\\mu}>40$ cm and $KE_p>40$ MeV\nGUMP $\\chi^2$",
     f'tot sel. {10272}\nsel. corr. {7909}\ntot true {16543}\neff: {0.47808740857160126*100:.2f} pur: {0.7699571651090342*100:.2f}',
     'tot sel. 44',
     f'tot sel. {2177}\nsel. corr. {1464}\ntot true {5342}\neff: {0.2740546611755897*100:.2f} pur: {0.6724850711988976*100:.2f}',
     'tot sel. 2'],

    ["$L_{\\mu}>40$ cm and $KE_p>40$ MeV\nGUMP $\\chi^2$\nno $p$ trkScore",
     f'tot sel. {9688}\nsel. corr. {7558}\ntot true {16543}\neff: {0.4568699752161035*100:.2f} pur: {0.7801403798513625*100:.2f}',
     'tot sel. 42',
     f'tot sel. {2022}\nsel. corr. {1367}\ntot true {5342}\neff: {0.2558966679146387*100:.2f} pur: {0.6760633036597429*100:.2f}',
     'tot sel. 2'],

    ["$L_{\\mu}>40$ cm and $KE_p>40$ MeV\nGUMP $\\chi^2$\nno $p$ trkScore\n$\\mu/p$ only",
     f'tot sel. {8984}\nsel. corr. {3899}\ntot true {9010}\neff: {0.43274139844617093*100:.2f} pur: {0.4339937666963491*100:.2f}',
     'tot sel. 37',
     f'tot sel. {1882}\nsel. corr. {1081}\ntot true {4192}\neff: {0.25787213740458015*100:.2f} pur: {0.5743889479277364*100:.2f}',
     'tot sel. 2'],

    ["$L_{\\mu}>50$ cm and $KE_p>50$ MeV\nGUMP $\\chi^2$\nno $p$ trkScore\n$\\mu/p$ only",
     f'tot sel. {8179}\nsel. corr. {3521}\ntot true {8130}\neff: {0.43308733087330875*100:.2f} pur: {0.43049272527203813*100:.2f}',
     'tot sel. 27',
     f'tot sel. {1615}\nsel. corr. {924}\ntot true {3408}\neff: {0.2711267605633803*100:.2f} pur: {0.5721362229102167*100:.2f}',
     'tot sel. 2']
]


table = plt.table(
    cellText=rows,
    colLabels=[
        "",
        r"MC $1\mu 1p+1\mu Np$",
        r"Offbeam $1\mu 1p+1\mu Np$",
        r"MC $1\mu Np$",
        r"Offbeam $1\mu Np$"
    ],
    cellLoc="center",
    loc="center",
    colWidths=[0.3, 0.21, 0.21, 0.21, 0.21]
)

table.auto_set_font_size(False)
table.set_fontsize(10)


# Altezza dinamica delle righe in base al contenuto
for i, row in enumerate(rows, start=1):

    # numero massimo di righe di testo presente nella riga
    n_lines = max(
        str(cell).count("\n") + 1
        for cell in row
    )

    for j in range(5):
        table[(i, j)].set_height(0.075 * n_lines)


# Altezza header
header_lines = max(
    str(cell).count("\n") + 1
    for cell in [
        "",
        r"MC $1\mu 1p+1\mu Np$",
        r"Offbeam $1\mu 1p+1\mu Np$",
        r"MC $1\mu Np$",
        r"Offbeam $1\mu Np$"
    ]
)

for j in range(5):
    table[(0, j)].set_height(0.06 * header_lines)


plt.show()